In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

from xgboost import XGBRegressor


In [9]:
df = pd.DataFrame(pd.read_csv('../data/preprocessed/model_data.csv'))

In [10]:
target_col = "future_rv_20d"

features = [
    "rv_5d",
    "rv_20d",
    "rv_60d",
    "vix_close"
]

required_cols = ["date", target_col] + features
df = df[required_cols].dropna().reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Date range:", df["date"].min(), df["date"].max())

Dataset shape: (3041, 6)
Date range: 2010-01-04 2022-10-26


In [11]:
n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

horizon = 20

train = df.iloc[:train_end - horizon].copy()
val = df.iloc[train_end:val_end - horizon].copy()
test = df.iloc[val_end:].copy()

In [16]:
sup_features = [
    "rv_5d",
    "rv_20d",
    "rv_60d"
]

X_train_sup = train[sup_features]
y_train = train[target_col]

X_val_sup = val[sup_features]
y_val = val[target_col]

X_test_sup = test[sup_features]

In [34]:
ridge_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=100))
])

In [35]:
ridge_pipeline.fit(X_train_sup, y_train)

ridge_val = ridge_pipeline.predict(X_val_sup)

print(
    "Ridge validation MAE:",
    mean_absolute_error(y_val, ridge_val)
)

Ridge validation MAE: 0.08456658898379518


Tested Alphas: 0.1, 1, 10, 100. 100 had lowest MAE